In [ ]:
import flow
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import shutil
import matplotlib.pyplot as plt
from flow import FlowProject, directives
from pymser import pymser
import warnings
import panedr

In [ ]:
from utils.molec_class_files import esolvs
dict = esolvs.make_dict()
gaff_param_set = dict["EG"].nm_kb_to_A_kJmol(dict["EG"].gaff_params)
#{293.15: 4.5421881351032553e-07, 303.15: 1.480803172240534e-06, 313.15: 4.424693883895992e-06, 323.15: 1.2225802187737217e-05, 333.15: 3.144734673775623e-05}

In [ ]:
file = "example_output_files/sft_calc.edr"
edr = panedr.edr_to_df(file)

In [ ]:
edr.tail()

In [ ]:
dt = edr["Time"].values
print(dt)
nsteps = 70000000
tot_rows = 70000
prod_step = 10000
nprod = tot_rows*prod_step
print(nprod)
print(nsteps/tot_rows)
print(nsteps/prod_step)

In [ ]:
len(edr)

In [ ]:
np.mean(edr["Pressure"].values)

In [ ]:
# Get the density and volume data
sim_name = "sft_calc"
# file = "example_output_files/" + sim_name + ".edr"
file = "density_iters/runs/workspace/c4e7cea1aaf17ab3c761d6d98f531e6e/inter_prod.edr"
df_all = panedr.edr_to_df(file)
print(df_all.columns)
eq_data_dict = {}
property = "#Surf*SurfTen"
if property in df_all.columns:
    df = df_all[["Time", property]].copy()
    print(df)

# elif property in ["Total Energy", "Volume", "Pressure", "#Surf*SurfTen"]:
    # command = "source /afs/crc.nd.edu/group/maginn/group_members/Ryan_DeFever/software/gromacs-2020/gromacs-dp/bin/GMXRC"
    # subprocess.run(command, shell=True, check=True)
    # print(os.path.exists(os.path.join("example_output_files", f"{sim_name}.edr")))
    # command = (
    #     f"source /afs/crc.nd.edu/group/maginn/group_members/Ryan_DeFever/software/gromacs-2020/gromacs-dp/bin/GMXRC && module load gromacs && gmx_d energy -f example_output_files/{sim_name}.edr -s example_output_files/{sim_name}.tpr -o {sim_name}_{property}.xvg"
    # )
    # subprocess.run(command, input=f"{property}\n", text=True, check=True, shell = True)
    # prop_data = np.loadtxt(
    #     sim_name + "_" + property + ".xvg", comments=["#", "@"]
    # )

    # df = pd.DataFrame(prop_data)

property_data = df.iloc[:, 1].values
print(np.mean(property_data))
plt.plot(df.iloc[:, 0].values, df.iloc[:, 1].values)
time_data = df.iloc[:, 0].values
eq_col_file = sim_name + "_" + property + ".csv"
eq_data_dict[property] = {
    "data": property_data,
    "time_data": time_data,
    "file": eq_col_file,
}

In [ ]:
def calc_mass_dens(density):
    #Find the region attributed to liquid density
    mass_dens_z = density[:, 1]
    find_liq_slab = find_bulk_liq_index(density)
    range_for_liq_dens = find_liq_slab[0]
    median_dens_liq = find_liq_slab[3]
    #Calculate the density
    prop_vals = mass_dens_z[range_for_liq_dens] #kg/m^3
    plt.scatter(density[:, 0], density[:, 1], label='ES', color='blue')
    plt.axhline(y=median_dens_liq, color='g', linestyle='--', label= 'Median')
    plt.xlabel('Z (nm)')
    plt.ylabel('Density (kg/m^3)')
    plt.scatter(density[range_for_liq_dens, 0], density[range_for_liq_dens, 1], label='ES', color='red')
    plt.show()
    return prop_vals
from scipy.signal import find_peaks
from findpeaks import findpeaks


def find_bulk_liq_index(density):
    #Use np.diff to approximate the 1st derivative
    ES_numdens_z = density[:,1]
    x = density[:,0]
    # dy = np.diff(ES_numdens_z)
    dy = np.gradient(ES_numdens_z, x)
    # dy = np.gradient(np.gradient(ES_numdens_z, x), x)
    # print(dy)
    # plt.scatter(np.arange(0,len(dy),1), dy, label='ES', color='blue')
    # plt.scatter(results['x'].iloc[interfaces], results['y'].iloc[interfaces], label='ES', color='red')
    #Use findpeaks to find the peaks and valleys, interpolating to get as close as possible
    fp = findpeaks(lookahead=1, interpolate=10)
    results = fp.fit(dy)["df_interp"]
    all_peaks = results[results['peak'] | results['valley']].index.values

    #get the highest peak and the lowest valley, these are the interfaces
    if_r = all_peaks[np.argsort(abs(results['y'].iloc[all_peaks]))[-2:]]
    

    #Divide the indices by 10 to get the correct index for the density based on interpolation
    interfaces = [int(i/10) for i in if_r]
    range_org_liq = np.arange(interfaces[0], interfaces[1], 1)
    #Get the median density from the bulk range
    median_dens_liq = np.median(ES_numdens_z[range_org_liq])

    plt.scatter(results['x']*max(x)/(len(x)*10), results['y'], label='ES', color='blue')
    plt.scatter(results['x'].iloc[if_r[0]:if_r[1]]*max(x)/(len(x)*10), results['y'].iloc[if_r[0]:if_r[1]], label='ES', color='red')
    
    plt.xlabel('Z (nm)')
    plt.ylabel('d_rho/dz (kg/m^3/nm)')
    plt.show()

    # Find the first and last index where density >= median
    valid_idx = np.where(ES_numdens_z[range_org_liq] >= median_dens_liq)[0]
    if valid_idx.size > 0:
        start = valid_idx[0]
        end = valid_idx[-1] + 1  # +1 to include the last valid index
        range_for_liq_dens = range_org_liq[start:end]
    else:
        range_for_liq_dens = np.array([], dtype=int)
    # print(range_org_liq)
    # range_for_liq_dens = range_org_liq
    # print(range_for_liq_dens)

    return range_for_liq_dens, interfaces, ES_numdens_z, median_dens_liq

# from scipy.signal import find_peaks
# def find_bulk_liq_index(density):
#     #Use np.diff to approximate the 1st derivative
#     ES_numdens_z = density[:,1]
#     x = density[:,0]
#     # dy = np.gradient(ES_numdens_z, x)
#     dy = np.gradient(np.gradient(ES_numdens_z, x), x)
#     # plt.scatter(np.arange(0,len(dy),1), dy, label='ES', color='blue')
#     # plt.scatter(results['x'].iloc[interfaces], results['y'].iloc[interfaces], label='ES', color='red')
#     #Use findpeaks to find the peaks and valleys, interpolating to get as close as possible
#     fp = findpeaks(lookahead=1, interpolate=10, params = {"height": 300})
#     results = fp.fit(dy)["df_interp"]
#     all_peaks = results[results['peak'] | results['valley']].index.values
    
#     #get the highest peak and the lowest valley, these are the interfaces
#     # all_peaks, _ = find_peaks(dy, height=300, threshold = 2, distance=50)
#     if_r = all_peaks#[np.argsort(abs(results['y'].iloc[all_peaks]))[-2:]]
#     if_r = [all_peaks[0], all_peaks[-1]]
    
#     plt.scatter(x, dy, label='ES', color='blue')
#     plt.scatter(x[if_r], dy[if_r], label='ES', color='red')

#     #Divide the indices by 10 to get the correct index for the density based on interpolation
#     interfaces = if_r
#     range_org_liq = np.arange(interfaces[0], interfaces[1], 1)
#     #Get the median density from the bulk range
#     median_dens_liq = np.median(ES_numdens_z[range_org_liq])

#     plt.xlabel('Z (nm)')
#     plt.ylabel('d_rho/dz (kg/m^3/nm)')
#     plt.show()

#     # Find the first and last index where density >= median
#     valid_idx = np.where(ES_numdens_z[range_org_liq] >= median_dens_liq)[0]
#     if valid_idx.size > 0:
#         start = valid_idx[0]
#         end = valid_idx[-1] + 1  # +1 to include the last valid index
#         range_for_liq_dens = range_org_liq[start:end]
#     else:
#         range_for_liq_dens = np.array([], dtype=int)
#     # print(range_org_liq)
#     # range_for_liq_dens = range_org_liq
#     # print(range_for_liq_dens)

#     return range_for_liq_dens, interfaces, ES_numdens_z, median_dens_liq

In [ ]:
density = np.loadtxt("density_iters/runs/workspace/c4e7cea1aaf17ab3c761d6d98f531e6e/inter_prod_density.xvg", comments=["#", "@"])
# density = np.loadtxt("example_output_files/R125_pure_295K_final_rhoMass_profile_System.xvg", comments=["#", "@"])


prop_vals = calc_mass_dens(density)

# print(prop_vals)


In [ ]:
def plot_res_pymser(t_col, eq_col, results, name):
    fig, [ax1, ax2] = plt.subplots(
        1, 2, gridspec_kw={"width_ratios": [2, 1]}, sharey=True
    )

    ax1.set_ylabel(name, color="black", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Time (ns)", fontsize=14, fontweight="bold")

    ax1.plot(t_col, eq_col, label="Raw data", color="blue")

    ax1.plot(
        t_col[results["t0"] :],
        results["equilibrated"],
        label="Equilibrated data",
        color="red",
    )

    ax1.plot(
        [0, t_col[-1]],
        [results["average"], results["average"]],
        color="green",
        zorder=4,
        label="Equilibrated average",
    )

    ax1.fill_between(
        t_col,
        results["average"] - results["uncertainty"],
        results["average"] + results["uncertainty"],
        color="lightgreen",
        alpha=0.3,
        zorder=4,
    )

    # ax1.set_yticks(np.arange(eq_col.min(), eq_col.max(), eq_col.max()/10))
    ax1.set_xlim(t_col.min(), t_col.max())
    ax1.tick_params(axis="y", labelcolor="black")

    ax1.grid(alpha=0.3)
    ax1.legend()

    ax2.hist(
        eq_col,
        orientation="horizontal",
        bins=30,
        edgecolor="blue",
        lw=1.5,
        facecolor="white",
        zorder=3,
    )

    bin_red = 10
    ax2.hist(
        results["equilibrated"],
        orientation="horizontal",
        bins=bin_red,
        edgecolor="red",
        lw=1.5,
        facecolor="white",
        zorder=3,
    )

    ymax = int(ax2.get_xlim()[-1])

    ax2.plot(
        [0, ymax],
        [results["average"], results["average"]],
        color="green",
        zorder=4,
        label="Equilibrated average",
    )

    ax2.fill_between(
        range(ymax),
        results["average"] - results["uncertainty"],
        results["average"] + results["uncertainty"],
        color="lightgreen",
        alpha=0.3,
        zorder=4,
    )

    ax2.set_xlim(0, ymax)

    ax2.grid(alpha=0.5, zorder=1)

    fig.set_size_inches(9, 5)
    fig.set_dpi(100)
    fig.tight_layout()
    # save_name = "MSER_eq_vol.png"
    # fig.savefig(job.fn(save_name), dpi=300, facecolor="white")
    # plt.close(fig)

def check_equil_converge(eq_data_dict, prod_tol):
    equil_matrix = []
    res_matrix = []
    prod_begin = 0
    prop_names = list(eq_data_dict.keys())
    num_cols = len(prop_names)

    try:
        # Load data for both boxes
        for key in list(eq_data_dict.keys()):
            eq_col = eq_data_dict[key]["data"]
            prod_tol = len(eq_col)/4
            print(prod_tol)
            batch_size = max(1, int(len(eq_col) * 0.0005))

            # Try with ADF test enabled, fallback without it if it fails
            try:
                results = pymser.equilibrate(
                    eq_col,
                    LLM=False,
                    batch_size=batch_size,
                    ADF_test=True,
                    uncertainty="uSD",
                    print_results=False,
                )
                adf_test_failed = results["critical_values"]["1%"] <= results["adf"]
            except:
                results = pymser.equilibrate(
                    eq_col,
                    LLM=False,
                    batch_size=batch_size,
                    ADF_test=False,
                    uncertainty="uSD",
                    print_results=False,
                )
                results["adf"], results["critical_values"], adf_test_failed = (
                    None,
                    None,
                    False,
                )

            equilibrium = len(eq_col) - results["t0"] >= prod_tol
            print(eq_col[results["t0"]])
            prod_begin = results["t0"]
            print("Results prod",results["t0"])
            equil_matrix.append(equilibrium and not adf_test_failed)
            res_matrix.append(results)

        for i, is_equilibrated in enumerate(equil_matrix):
            key_name = list(eq_data_dict.keys())[i]
            col_vals = eq_data_dict[key_name]["data"]
            t_vals = eq_data_dict[key_name]["time_data"]
            print(t_vals.min(), t_vals.max())
            # plot all

            # if not all(equil_matrix):
            plot_res_pymser(
                t_vals, col_vals, res_matrix[i], prop_names[i % num_cols]
            )

            # Display outcome
            prod_cycles = len(col_vals) - res_matrix[i]["t0"]
            if is_equilibrated:
                # Plot successful equilibration
                statement = f"       > Success! Found {prod_cycles} production cycles."
            else:
                # Plot failed equilibration
                statement = f"       > Equil Failure! "
                if res_matrix[i]["adf"] is None:
                    # Note: ADF test failed to complete
                    statement += f"ADF test failed to complete! "
                elif res_matrix[i]["adf"] > res_matrix[i]["critical_values"]["1%"]:
                    adf, one_pct = (
                        res_matrix[i]["adf"],
                        res_matrix[i]["critical_values"]["1%"],
                    )
                    statement += f"ADF value: {adf}, 99% confidence value: {one_pct}! "
                if len(col_vals) - res_matrix[i]["t0"] < prod_tol:
                    statement += f"Only {prod_cycles} production cycles found."
            print(statement)
            print("All matrix eq", all(equil_matrix))
            return prod_begin
            # with open("Equil_Output.txt", "a") as f:
            #     print(statement, file=f)

    except Exception as e:
        # This will cause an error in the GEMC operation which lets us know that the job failed
        raise Exception("Error processing job ")

    



In [ ]:
# print(df["Time"].values)
prod_begin = check_equil_converge(eq_data_dict, prod_tol = None)
# df = panedr.edr_to_df(file)
# volume = df[property].values
# vol_prod = volume[prod_begin:]
# print(vol_prod)

In [ ]:
from thermo import Chemical
from thermo import *
import unyt as u
from utils.molec_class_files import esolvs

# Load class properies for each training molecule
name = "DMF"
mol_names = [name]  # , "Gly", "ACN", "MeOH", "DMSO", "THF", "DCM", "DEC"]
molec_dict = esolvs.make_dict(mol_names)
molec_data = molec_dict[name]
# from thermo.eos import PengRobinson

# Define the compound (e.g., 'methane', 'ethane', etc.)
smiles_string =  molec_data.smiles_str # Replace with your compound of interest
T = list(molec_data.expt_Pvap.keys())[0]  # Temperature in Kelvin (adjust as needed)
P = list(molec_data.expt_Pvap.values())[0] # Pressure in Pascals (adjust as needed)
P = float((P*u.bar).in_units(u.Pa).value)
# Create a Chemical object for the compound
constants = Chemical(smiles_string)
# print(constants)
# eos_kwargs = {'Pc': constants.Pc, 'Tc': constants.Tc, 'omega': constants.omega}

eos = PR(constants.Tc, constants.Pc, constants.omega, T=T, P=P)

# Use the Peng-Robinson EOS to calculate the vapor density
molar_vol_g = eos.V_g
vapor_density = molec_data.molecular_weight/1000/ molar_vol_g
# Print the vapor density
print(f'PR Vapor Density of {constants.name} at {T} K and {P} Pa: rho_v = {vapor_density} kg/m^3')

unit_T = T * u.K
mw = molec_data.molecular_weight * (u.g / u.mol)
R = 8.314 * (u.Pa * u.m**3) / (u.mol * u.K)
vapor_density = float(
    ((molec_data.expt_Pvap[T] * u.bar * mw) / (R * unit_T)).in_units("kg/m**3").value
)
print(f'IG Vapor Density of {constants.name} at {T} K and {P} Pa: rho_v = {vapor_density} kg/m^3')